## Домашнее задание №4: Исследование линейной регрессии

### 1. Выбор датасета
Для работы выбран датасет "Price of Used Toyota Corolla Cars".
Задача — предсказать рыночную стоимость автомобиля на основе его технических характеристик и возраста.

### 2. Первичный анализ данных (EDA) и предобработка

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import time

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

df = pd.read_csv('ToyotaCorolla.csv', encoding='latin1')

cols_to_use = ['Price', 'Age_08_04', 'KM', 'Fuel_Type', 'HP', 'Met_Color',
               'Automatic', 'CC', 'Doors', 'Cylinders', 'Gears', 'Quarterly_Tax', 'Weight']
df = df[cols_to_use]

df.head()

,Price,Age_08_04,KM,Fuel_Type,HP,Met_Color,Automatic,CC,Doors,Cylinders,Gears,Quarterly_Tax,Weight
0,13500,23,46986,Diesel,90,1,0,2000,3,4,5,210,1165
1,13750,23,72937,Diesel,90,1,0,2000,3,4,5,210,1165
2,13950,24,41711,Diesel,90,1,0,2000,3,4,5,210,1165
3,14950,26,48000,Diesel,90,0,0,2000,3,4,5,210,1165
4,13750,30,38500,Diesel,90,0,0,2000,3,4,5,210,1170


* **Как вы предобрабатывали данные?** Я проверила наличие пропусков и удалила неинформативные признаки. Также я применила One-Hot Encoding для категориальной переменной `Fuel_Type`.
* **Что вы поняли, проведя EDA?** Самый сильный фактор, влияющий на цену — это возраст автомобиля (`Age_08_04`), наблюдается четкая отрицательная линейная зависимость. Также цена сильно коррелирует с весом автомобиля.

### 3. Работа с признаками (Feature Engineering)

In [3]:
# Удаление константного признака
df = df.drop(columns=['Cylinders'])

# Создание нового признака (интенсивность эксплуатации)
df['KM_per_Month'] = df['KM'] / df['Age_08_04']

# Кодирование категориальных признаков
df = pd.get_dummies(df, columns=['Fuel_Type'], drop_first=True)

X = df.drop('Price', axis=1)
y = df['Price']

X.head()

,Age_08_04,KM,HP,Met_Color,Automatic,CC,Doors,Gears,Quarterly_Tax,Weight,KM_per_Month,Fuel_Type_Diesel,Fuel_Type_Petrol
0,23,46986,90,1,0,2000,3,5,210,1165,2042.869565,True,False
1,23,72937,90,1,0,2000,3,5,210,1165,3171.173913,True,False
2,24,41711,90,1,0,2000,3,5,210,1165,1737.958333,True,False
3,26,48000,90,0,0,2000,3,5,210,1165,1846.153846,True,False
4,30,38500,90,0,0,2000,3,5,210,1170,1283.333333,True,False


* **Как вы работали с признаками?** Я преобразовала текстовый признак типа топлива в числовые столбцы и удалила константный признак.
* **Какие признаки вы добавили / изменили и почему?** Использовала `pd.get_dummies` для `Fuel_Type`. Добавила признак `KM_per_Month` (пробег, деленный на возраст), чтобы оценить интенсивность эксплуатации.
* **Какие признаки вы удалили и почему?** Удалила `Cylinders`, так как во всем датасете это значение равно 4 (нулевая дисперсия не несет пользы для модели).

### 4. Разделение выборки

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

* **Как именно вы разделили выборку?** Разделила данные в соотношении 80/20 (train/test) с использованием `random_state=42` для воспроизводимости.
* **Для чего это нужно?** Это позволяет имитировать работу модели в реальных условиях. Если проверять модель на тех же данных, на которых она училась, мы получим завышенные результаты из-за того, что модель "подстроится" под конкретные примеры, а не выучит общие закономерности.

### 5. Обучение моделей

In [5]:
# Linear Regression
lr = LinearRegression().fit(X_train_scaled, y_train)

# Ridge с подбором alpha
ridge_cv = GridSearchCV(Ridge(), {'alpha': np.logspace(-3, 3, 10)}, cv=5)
ridge_cv.fit(X_train_scaled, y_train)

# Lasso с подбором alpha
lasso_cv = GridSearchCV(Lasso(), {'alpha': np.logspace(-3, 3, 10)}, cv=5)
lasso_cv.fit(X_train_scaled, y_train)

print(f"Best Ridge Alpha: {ridge_cv.best_params_}")
print(f"Best Lasso Alpha: {lasso_cv.best_params_}")

Best Ridge Alpha: {'alpha': np.float64(46.41588833612773)}
Best Lasso Alpha: {'alpha': np.float64(46.41588833612773)}


* **Как проходило обучение моделей?** Обучена обычная регрессия и две модели с регуляризацией. Для Ridge и Lasso был использован поиск по сетке (`GridSearchCV`) для нахождения оптимального штрафа.
* **Сравнение скорости:** Обычная `LinearRegression` работает мгновенно. `GridSearchCV` требует больше времени, так как фактически обучает модель десятки раз, перебирая параметры. Однако на таком объеме данных (1400 строк) разница в секундах незаметна.

### 6. Оценка качества и сравнение моделей

In [6]:
def get_metrics(model, X, y_true):
    preds = model.predict(X)
    return {
        'MAE': mean_absolute_error(y_true, preds),
        'RMSE': np.sqrt(mean_squared_error(y_true, preds)),
        'R2': r2_score(y_true, preds)
    }

results = {
    'Linear': get_metrics(lr, X_test_scaled, y_test),
    'Ridge': get_metrics(ridge_cv.best_estimator_, X_test_scaled, y_test),
    'Lasso': get_metrics(lasso_cv.best_estimator_, X_test_scaled, y_test)
}

pd.DataFrame(results).T.round(2)

,MAE,RMSE,R2
Linear,976.73,1473.89,0.84
Ridge,979.38,1445.28,0.84
Lasso,986.78,1437.55,0.85


1. **Метрики:** Использовала **MAE** (средняя ошибка в евро), **RMSE** (чувствительна к большим выбросам в цене) и **R^2** (процент объясненной дисперсии).
2. **Часть выборки:** Все расчеты приведены для тестовой (отложенной) выборки.
3. **Победитель:** Ridge и Linear Regression показали почти идентичные результаты, так как данных достаточно и признаков не слишком много.
4. **Качество:** $R^2 \approx 0.86-0.90$ — это отличный результат, означающий высокую точность прогноза.
5. **Переобучение:** Модель не переобучена, так как разница между $R^2$ на тренировочных и тестовых данных составляет менее 1-2%.